# **MoE并行策略**

**导航**：[← 上一章：MoE架构概述](02.02_moe_architexture_overview.ipynb) | [返回课程主页](02.01_chapter_intro.ipynb) | [下一章：Dispatch&Combine算子 →](02.04_dispatch_combine_operator.ipynb)

---

本章将介绍MoE模型在分布式训练中的并行策略，内容涵盖数据并行、模型并行、专家并行及其组合策略。

**本章目录**（点击跳转）：
- [分布式训练基础回顾](#1-分布式训练基础回顾)
- [MoE并行策略概览](#2-moe并行策略概览)
- [专家并行（Expert Parallelism）](#3-专家并行expert-parallelism)
- [数据并行 + 专家并行](#4-数据并行--专家并行)
- [张量并行 + 专家并行](#5-张量并行--专家并行)
- [流水线并行与MoE](#6-流水线并行与moe)
- [通信分析与优化](#7-通信分析与优化)
- [MC2融合算子在MoE并行中的应用](#8-mc2融合算子在moe并行中的应用)
- [课后练习](#课后练习)

---
## **1. 分布式训练基础回顾**
在深入MoE并行策略之前，先回顾分布式训练的核心概念。

> **前置知识**：本章需要了解MoE的基本架构，建议先阅读 [02.02 MoE架构概述](02.02_moe_architexture_overview.ipynb)

### **并行策略分类**

<table style="margin: 0; margin-right: auto; border-collapse: collapse;">
  <tr>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">策略</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">切分维度</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">通信类型</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">适用场景</th>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">数据并行（DP）</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">数据</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">AllReduce</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">小模型、大数据</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">张量并行（TP）</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">模型权重</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">AllReduce/AllGather</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">大模型单层</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">流水线并行（PP）</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">模型层</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">P2P</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">大模型多层</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">专家并行（EP）</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">专家</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">AllToAll</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">MoE模型</td>
  </tr>
</table>

### **常用集合通信操作**

<table style="margin: 0; margin-right: auto; border-collapse: collapse;">
  <tr>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">操作</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">输入</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">输出</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">功能说明</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">MoE应用场景</th>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><strong>AllReduce</strong></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">各NPU数据</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">所有NPU相同结果</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">求和并广播到所有设备</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">梯度同步、TP同步</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><strong>AllGather</strong></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">各NPU数据</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">所有NPU完整数据</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">每NPU收集所有设备数据</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">门控结果广播</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><strong>ReduceScatter</strong></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">各NPU数据</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">各NPU分片结果</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">求和并分片输出</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">减少输出数据量</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><strong>AllToAll</strong></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">各NPU分片数据</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">各NPU重组数据</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">每NPU发送不同数据给不同设备</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">Token分发/聚合</td>
  </tr>
</tr>

#### **AllReduce: 所有设备求和并广播结果**

<img src="./images/02.03_AllReduce.png" width="800">

#### **AllGather: 每设备收集所有设备数据**

<img src="./images/02.03_AllGather.png" width="800">

#### **ReduceScatter: AllReduce + 分片输出**

<img src="./images/02.03_ReduceScatter.png" width="800">

#### **AllToAll: 每设备发送不同数据给不同设备**

<img src="./images/02.03_AllToAll.png" width="800">

---
## **2. MoE并行策略概览**
MoE模型引入了独特的并行维度——专家并行，需要与现有的数据并行、张量并行、流水线并行进行组合。

### **MoE并行策略组合矩阵**

<table style="margin: 0; margin-right: auto; border-collapse: collapse;">
  <tr>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">组合</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">通信模式</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">通信量</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">复杂度</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">适用规模</th>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">DP + EP</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">AllToAll</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">中等</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">低</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">中小规模MoE</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">TP + EP</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">AllToAll + AllReduce</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">高</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">中</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">大规模MoE</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">DP + TP + EP</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">AllToAll + AllReduce</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">高</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">高</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">超大规模MoE</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">PP + EP</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">AllToAll + P2P</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">中等</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">中</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">深层MoE</td>
  </tr>
</table>

---
## **3. 专家并行（Expert Parallelism）**
专家并行是MoE模型独有的并行策略，将不同专家放置在不同设备上。

### **专家并行原理**

#### **4专家 × 4NPU 专家分布**

<img src="./images/02.03_experts_distribute.png" width="800">

#### **Token分布流程**

<img src="./images/02.03_token_distribution_process.png" width="400">

### **专家并行通信分析**

<table style="margin: 0; margin-right: auto; border-collapse: collapse;">
  <tr>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">通信阶段</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">通信类型</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">数据量</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">频率</th>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">Token分发</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">AllToAll</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">Token数 × Hidden × 2</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">每MoE层</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">结果聚合</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">AllToAll</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">Token数 × Hidden × 2</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">每MoE层</td>
  </tr>
</table>

### **专家并行参数配置**

<table style="margin: 0; margin-right: auto; border-collapse: collapse;">
  <tr>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">参数</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">说明</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">示例值</th>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>num_experts</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">专家总数</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">8</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>expert_parallel_size</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">专家并行度（NPU数量）</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">4</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>experts_per_npu</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">每NPU放置的专家数</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">2</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"><code>top_k</code></td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">每Token激活专家数</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">2</td>
  </tr>
</table>

<table style="margin: 0; margin-right: auto; border-collapse: collapse;">
  <tr>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">NPU</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">放置专家</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">说明</th>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">NPU0</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">Expert 0, Expert 1</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">experts_per_npu = num_experts / expert_parallel_size</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">NPU1</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">Expert 2, Expert 3</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"></td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">NPU2</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">Expert 4, Expert 5</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"></td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">NPU3</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">Expert 6, Expert 7</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;"></td>
  </tr>
</table>

---
## **4. 数据并行 + 专家并行**
数据并行与专家并行组合是最常见的MoE分布式策略。

### **DP + EP 架构**

<img src="./images/02.03_DP+EP_architecture.png" width="800">

**说明**：4卡 DP=2, EP=2 示例

### **通信模式**

<table style="margin: 0; margin-right: auto; border-collapse: collapse;">
  <tr>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">阶段</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">通信组</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">通信类型</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">说明</th>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">MoE层Token分发</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">EP组</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">AllToAll</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">专家间Token交换</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">MoE层结果聚合</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">EP组</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">AllToAll</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">专家结果返回</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">非MoE层梯度同步</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">DP组</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">AllReduce</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">梯度平均</td>
  </tr>
</table>

### **DP + EP 配置示例**

<table style="margin: 0; margin-right: auto; border-collapse: collapse;">
  <tr>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">配置项</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">值</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">说明</th>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">总NPU数</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">8</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">DP × EP</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">DP size</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">2</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">数据并行度</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">EP size</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">4</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">专家并行度</td>
  </tr>
</table>

<table style="margin: 0; margin-right: auto; border-collapse: collapse;">
  <tr>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">DP Group</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">EP组成员</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">说明</th>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">DP Group 0</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">[NPU0, NPU1, NPU2, NPU3]</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">EP组内AllToAll通信</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">DP Group 1</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">[NPU4, NPU5, NPU6, NPU7]</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">EP组内AllToAll通信</td>
  </tr>
</table>

<table style="margin: 0; margin-right: auto; border-collapse: collapse;">
  <tr>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">NPU</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">放置专家</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">DP组</th>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">NPU0, NPU4</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">Expert 0, 1</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">DP Group 0, 1</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">NPU1, NPU5</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">Expert 2, 3</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">DP Group 0, 1</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">NPU2, NPU6</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">Expert 4, 5</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">DP Group 0, 1</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">NPU3, NPU7</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">Expert 6, 7</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">DP Group 0, 1</td>
  </tr>
</table>

```
DP组间AllReduce同步梯度，EP组内AllToAll分发Token
```
---

## **5. 张量并行 + 专家并行**
当单个专家太大无法放入单个NPU时，需要使用张量并行切分专家。

### **专家张量并行切分**

**专家FFN切分方式 - 列并行 + 行并行：**

<img src="./images/02.03_tensor_parallel_segmentation.png" width="1200">
---

## **6. 流水线并行与MoE**
流水线并行将模型按层切分到不同设备，对MoE有特殊处理。

### **PP + MoE 架构**

**4卡 PP=4 示例：**

<img src="./images/02.03_PP+MOE_architecture.png" width="1200">

**MoE层内部处理：**

<img src="./images/02.03_moe_layer_processing.png" width="1200">

**说明**：如有EP，则存在本地通信。

### **PP + MoE 通信分析**

<table style="margin: 0; margin-right: auto; border-collapse: collapse;">
  <tr>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">通信类型</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">方向</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">触发时机</th>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">P2P发送</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">前向</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">层间激活传递</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">P2P接收</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">前向</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">层间激活接收</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">AllToAll</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">前向</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">MoE层内</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">P2P发送</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">反向</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">梯度传递</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">P2P接收</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">反向</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">梯度接收</td>
  </tr>
</table>

---
## **7. 通信分析与优化**
深入分析MoE各并行策略的通信开销与优化方法。

### **通信优化策略**

<table style="margin: 0; margin-right: auto; border-collapse: collapse;">
  <tr>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">优化方法</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">原理</th>
    <th style="border:1px solid #ccc; padding:8px; text-align:left;">效果</th>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">通信计算重叠</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">AllToAll与专家计算并行</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">隐藏通信延迟</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">容量因子优化</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">控制每个专家处理Token数</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">平衡负载</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">Token丢弃</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">超容量Token直接丢弃</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">减少通信</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">拓扑感知</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">按网络拓扑排列专家</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">减少跨节点通信</td>
  </tr>
  <tr>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">批处理优化</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">合并多个micro-batch</td>
    <td style="border:1px solid #ccc; padding:8px; text-align:left;">提高通信效率</td>
  </tr>
</table>

---
## **8. MC2融合算子在MoE并行中的应用**
MC2融合算子可以显著优化MoE分布式训练中的通信开销。

### **MoE关键通信点与MC2优化**

**传统模式：**

<img src="./images/02.03_traditional_model.png" width="800">

**MC2融合模式：**

<img src="./images/02.03_mc2_fusion_mode.png" width="800">
---

# **课后练习**
请根据本章内容认真思考，选出正确选项。  

1. （单选题）专家并行（EP）的核心思想是什么？  
    A. 将数据切片到不同NPU。    
    B. 将不同专家放置在不同NPU上。     
    C. 将模型层切片到不同NPU。    
    D. 将专家权重切片到不同NPU。  


2. （单选题）MoE中使用AllToAll通信的主要目的是什么？  
    A. 同步梯度。  
    B. 在专家间分发和聚合Token。   
    C. 广播模型参数。  
    D. 执行AllReduce操作。    

3. （单选题）当单个专家太大无法放入单个NPU时，应采用什么策略？  
    A. 增加数据并行度。  
    B. 减少Top-K值。   
    C. 使用张量并行切分专家。  
    D. 减少专家数量。    

4. （单选题）MC2融合算子如何优化MoE通信？    
    A. 消除所有通信。  
    B. 增加Token丢弃率。   
    C. 减少专家数量。  
    D. 通过计算通信重叠隐藏通信延迟。   

5. （多选题）关于MoE并行策略，以下说法正确的是？    
    A. DP+EP组合中，EP组内使用AllToAll通信。   
    B. TP+EP组合中，TP组内使用AllReduce同步。   
    C. 流水线并行与MoE不兼容。  
    D. 通信量与Token数、Hidden维度、Top-K成正比。    

# **查看答案**

In [ ]:
!cat answer/02.03_answer.txt

---

**导航**：[← 上一章：MoE架构概述](02.02_moe_architexture_overview.ipynb) | [返回课程主页](02.01_chapter_intro.ipynb) | [下一章：Dispatch&Combine算子 →](02.04_dispatch_combine_operator.ipynb)

---